# 0. Load dữ liệu

- load `station_hour.csv` vào `df`
- load `stations.csv` vào `df_stations`

In [2]:
# load
PATH = '../data/raw/'

import pandas as pd
from pathlib import Path

# https://cpcb.gov.in/openpdffile.php?id=TGF0ZXN0RmlsZS8yNjRfMTU2NTI1Nzg1OV9tZWRpYXBob3RvMTU0MDcucGRm
df = pd.read_csv(
    PATH + "station_hour.csv",
    parse_dates=["Datetime"] # Nhớ parse date để tránh lag nếu vẽ đồ thị có date mà chưa parse
)
df_raw = df.copy()
print("station_hour.csv")
print([col for col in df.columns])
print()

print("stations.csv")
df_stations = pd.read_csv(PATH + "stations.csv")
print([col for col in df_stations.columns])

C:\Users\Admin\AppData\Local\Temp\ipykernel_16048\2139656056.py:8: DtypeWarning: Columns (15: AQI_Bucket) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


station_hour.csv
['StationId', 'Datetime', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI', 'AQI_Bucket']

stations.csv
['StationId', 'StationName', 'City', 'State', 'Status']


## 3. Cleanning

### 3.1. Kiểm tra tổng thể

In [3]:
import pandas as pd
# Các cột cần kiểm tra
pollutants = [
    'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3',
    'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene'
]
numeric_cols = [c for c in pollutants + ['AQI'] if c in df.columns]
n = len(df)
# 1. Trùng toàn bộ dòng: đếm các dòng trùng dư, không tính lần xuất hiện đầu
duplicate_rows = int(df.duplicated().sum())
# 2. Datetime không hợp lệ
dt_parsed = pd.to_datetime(df['Datetime'], errors='coerce')
invalid_datetime = (
    df['Datetime'].notna() & dt_parsed.isna()
)
invalid_datetime_count = int(invalid_datetime.sum())
# 3. Trùng thời gian trong cùng trạm
station_key = df['StationId'].astype('string').str.strip()
valid_pair = (
    station_key.notna()
    & station_key.ne('')
    & dt_parsed.notna()
)
pairs = pd.DataFrame({
    'StationId': station_key.loc[valid_pair],
    'Datetime': dt_parsed.loc[valid_pair]
})
duplicate_datetime = int(
    pairs.duplicated(['StationId', 'Datetime']).sum()
)
# 4. Các ô không chuyển được sang kiểu số

invalid_numeric = pd.DataFrame(index=df.index)
for col in numeric_cols:
    converted = pd.to_numeric(df[col], errors='coerce')
    invalid_numeric[col] = df[col].notna() & converted.isna()
invalid_numeric_count = int(invalid_numeric.sum().sum())
# 5. StationId không tồn tại trong df_stations
valid_station_ids = (
    df_stations['StationId']
    .astype('string')
    .str.strip()
    .dropna()
)
invalid_station = (
    station_key.isna()
    | station_key.eq('')
    | ~station_key.isin(valid_station_ids)
).fillna(True)
invalid_station_count = int(invalid_station.sum())
# 6. Số lượng và tỷ lệ dữ liệu thiếu
missing_count = int(df.isnull().sum().sum())
missing_total = int(df.size)
# 7. Lập bảng tổng hợp
checks = [
    ('Trùng toàn bộ dòng', duplicate_rows, n),
    ('Trùng StationId + Datetime', duplicate_datetime, n),
    ('StationId không thuộc df_stations', invalid_station_count, n),
    ('Ô Datetime sai định dạng', invalid_datetime_count, n),
    (
        'Ô cột số không chuyển được sang numeric',
        invalid_numeric_count,
        n * len(numeric_cols)
    ),
    (
        'Tổng ô sai kiểu dữ liệu',
        invalid_datetime_count + invalid_numeric_count,
        n * (1 + len(numeric_cols))
    ),
    ('Dữ liệu thiếu', missing_count, missing_total)
]
summary = pd.DataFrame(
    checks,
    columns=['Vấn đề', 'Số lượng', 'Mẫu số tính tỷ lệ']
)
# Tính tỷ lệ phần trăm
summary['Tỷ lệ (%)'] = (
    summary['Số lượng']
    .div(summary['Mẫu số tính tỷ lệ'].replace(0, pd.NA))
    .mul(100)
    .round(3)
)
# Hiển thị bảng kết quả
display(summary)

,Vấn đề,Số lượng,Mẫu số tính tỷ lệ,Tỷ lệ (%)
0,Trùng toàn bộ dòng,0,2589083,0.000
1,Trùng StationId + Datetime,0,2589083,0.000
2,StationId không thuộc df_stations,0,2589083,0.000
3,Ô Datetime sai định dạng,0,2589083,0.000
4,Ô cột số không chuyển được sang numeric,0,33658079,0.000
5,Tổng ô sai kiểu dữ liệu,0,36247162,0.000
6,Dữ liệu thiếu,11664492,41425328,28.158


Nhận xét : Qua bước đầu kiểm tra, dữ liệu có tỉ lệ sạch cao, ít bị các lỗi về định dạng,và thông tin. Tuy nhiên vẫn có tỷ lệ thiếu dữ liệu lên đến 28%

### 3.2.1. giá trị trần
- Giá trị trần gần với khoảng giới hạn đo của máy, [theo như tài liệu (trang 18-19)](https://cpcb.gov.in/openpdffile.php?id=TGF0ZXN0RmlsZS8yNjRfMTU2NTI1Nzg1OV9tZWRpYXBob3RvMTU0MDcucGRm)

In [ ]:
# Các giá trị cao xuất hiện nhiều bất thường ơ PM2.5 và PM10
import matplotlib.pyplot as plt

top_pm10 = df[df['PM10'] >= 900]['PM10'].value_counts().head(10)
display(top_pm10)
print()

# plt.figure(figsize=(8, 4))
# top_pm10.plot(kind='bar', color='crimson', edgecolor='black')
# plt.title('Tần suất xuất hiện của các giá trị cực trị (>= 900)', fontsize=13)
# plt.xlabel('Giá trị PM10', fontsize=11)
# plt.ylabel('Số lần xuất hiện', fontsize=11)
# plt.xticks(rotation=45)
# plt.grid(axis='y', linestyle='--', alpha=0.7)
# plt.show()

# Lọc các giá trị cao và đếm top giá trị xuất hiện nhiều nhất
top_pm25 = df[df['PM2.5'] >= 900]['PM2.5'].value_counts().head(10)
display(top_pm25)
print()


# plt.figure(figsize=(8, 4))
# top_pm25.plot(kind='bar', color='crimson', edgecolor='black')
# plt.title('Tần suất xuất hiện của các giá trị cực trị (>= 900)', fontsize=13)
# plt.xlabel('Giá trị PM2.5', fontsize=11)
# plt.ylabel('Số lần xuất hiện', fontsize=11)
# plt.xticks(rotation=45)
# plt.grid(axis='y', linestyle='--', alpha=0.7)
# plt.show()

PM10
999.99     236
1000.00    189
985.00      77
947.00      42
994.50      41
989.00      29
923.00      26
983.00      25
918.00      24
900.00      23
Name: count, dtype: int64

PM2.5
999.99     478
995.00     213
985.00     188
1000.00    182
912.08      10
919.33       7
919.00       6
960.00       5
903.00       4
953.00       4
Name: count, dtype: int64

In [ ]:
def show_mask_rate(mask, df, cols):
    summary = pd.DataFrame({
        'Số lượng đánh dấu': mask[cols].sum(),
        'Tổng giá trị hợp lệ': df[cols].notna().sum()
    })

    summary['Tỷ lệ (%)'] = (
        summary['Số lượng đánh dấu']
        / summary['Tổng giá trị hợp lệ']
        * 100
    ).round(3)

    display(summary)

import numpy as np
import pandas as pd

pollutants = [
    'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3',
    'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene'
]

# Mask tổng thể, chỉ gồm các ô cần đánh dấu
mask1 = pd.DataFrame(False, index=df.index, columns=pollutants)

mask1['PM2.5'] = df['PM2.5'].isin([999.99, 1000, 995.0, 985.0])
mask1['PM10'] = df['PM10'].isin([999.99, 1000])

print(f'Số giá trị bị đánh dấu: {mask1.sum().sum():,}')
show_mask_rate(mask1, df, pollutants)

Số giá trị bị đánh dấu: 1,486


,Số lượng đánh dấu,Tổng giá trị hợp lệ,Tỷ lệ (%)
PM2.5,1061,1941394,0.055
PM10,425,1469831,0.029
NO,0,2035372,0.000
NO2,0,2060110,0.000
NOx,0,2098275,0.000
NH3,0,1352465,0.000
CO,0,2089781,0.000
SO2,0,1846346,0.000
O3,0,1863110,0.000
Benzene,0,1727504,0.000


In [24]:
# Xóa giá trị bị đánh dấu
df.loc[:, pollutants] = df[pollutants].mask(mask1)

### 3.2.2. Giá trị 0 liên tiếp
- các đoạn giá trị 0 dài bất thường và thường đi cùng nhau.

In [25]:
# > 7d

df['Datetime'] = pd.to_datetime(df['Datetime'])

mask2 = pd.DataFrame(False, index=df.index, columns=pollutants)
duration = pd.Timedelta('7d')

# Dùng vị trí dòng để tránh lỗi khi index bị trùng
for _, group in df.groupby('StationId', sort=False):
    group = group.sort_values('Datetime')
    idx = group.index

    for col in pollutants:
        is_zero = group[col].eq(0).fillna(False)

        # Mỗi chuỗi True liên tiếp có một mã riêng
        run_id = is_zero.ne(is_zero.shift()).cumsum()

        for _, run in group[is_zero].groupby(run_id[is_zero]):
            if run['Datetime'].max() - run['Datetime'].min() >= duration:
                mask2.loc[run.index, col] = True

print(f'Số giá trị 0 bị đánh dấu: {mask2.sum().sum():,}')
show_mask_rate(mask2, df, pollutants)

C:\Users\Shiichan\AppData\Local\Temp\ipykernel_13712\2042065890.py:6: Pandas4Warning: 'd' is deprecated and will be removed in a future version. Please use 'D' instead of 'd'.
  duration = pd.Timedelta('7d')


Số giá trị 0 bị đánh dấu: 603,921


,Số lượng đánh dấu,Tổng giá trị hợp lệ,Tỷ lệ (%)
PM2.5,0,1940333,0.000
PM10,0,1469406,0.000
NO,0,2035372,0.000
NO2,0,2060110,0.000
NOx,71410,2098275,3.403
NH3,0,1352465,0.000
CO,92745,2089781,4.438
SO2,0,1846346,0.000
O3,0,1863110,0.000
Benzene,185247,1727504,10.723


In [26]:
# Xóa giá trị bị đánh dấu
df.loc[:, pollutants] = df[pollutants].mask(mask2)

### 3.2.3. các đỉnh
- Các outlier tìm theo phương pháp IQR với k = 3, sliding window 48h

In [27]:

window = '48h'
min_periods = 24
k = 3

mask3 = pd.DataFrame(False, index=df.index, columns=pollutants)

for _, group in df.groupby('StationId', sort=False):
    group = group.sort_values('Datetime')

    for col in pollutants:
        s = group.set_index('Datetime')[col]
        q1 = s.rolling(window, center=True, min_periods=min_periods).quantile(0.25)
        q3 = s.rolling(window, center=True, min_periods=min_periods).quantile(0.75)

        iqr = q3 - q1
        lower = q1 - k * iqr
        upper = q3 + k * iqr

        spike = (s < lower) | (s > upper)

        # Gán mask theo thứ tự thời gian
        mask3.loc[group.sort_values('Datetime').index, col] = (
            spike.fillna(False).to_numpy()
        )

print(f'Số giá trị spike bị đánh dấu: {mask3.sum().sum():,}')
show_mask_rate(mask3, df, pollutants)

Số giá trị spike bị đánh dấu: 263,982


,Số lượng đánh dấu,Tổng giá trị hợp lệ,Tỷ lệ (%)
PM2.5,10599,1940333,0.546
PM10,6746,1469406,0.459
NO,59124,2035372,2.905
NO2,15727,2060110,0.763
NOx,26616,2026865,1.313
NH3,10101,1352465,0.747
CO,25207,1997036,1.262
SO2,21459,1846346,1.162
O3,15666,1863110,0.841
Benzene,32503,1542257,2.107


In [28]:
import numpy as np
import plotly.graph_objects as go

# Chọn trạm cần vẽ
station_id = df["StationId"].dropna().unique()[0]  # Đổi thành StationId bạn muốn

plot_df = df.loc[df["StationId"] == station_id].sort_values("Datetime")
spike = mask3.loc[plot_df.index, "PM2.5"].fillna(False).to_numpy()

fig = go.Figure(go.Scattergl(
    x=plot_df["Datetime"],
    y=plot_df["PM2.5"],
    mode="lines+markers",
    line=dict(color="steelblue", width=1),
    marker=dict(
        color=np.where(spike, "red", "steelblue"),
        size=5
    ),
    customdata=np.where(spike, "Spike", "Bình thường"),
    hovertemplate=(
        "Thời gian: %{x}<br>"
        "PM2.5: %{y}<br>"
        "Trạng thái: %{customdata}<extra></extra>"
    )
))

fig.update_layout(
    title=f"PM2.5 và các điểm spike — StationId: {station_id}",
    xaxis_title="Thời gian",
    yaxis_title="PM2.5",
    template="plotly_white",
    hovermode="closest",
    dragmode="zoom"
)

fig.update_xaxes(rangeslider_visible=True)
fig.show()

In [29]:
# Xóa giá trị bị đánh dấu
df.loc[:, pollutants] = df[pollutants].mask(mask3)

### 3.2.4. impute

In [30]:
# Tạo mask để bỏ qua những đoạn thiếu > 30%

import pandas as pd
import numpy as np

df['Datetime'] = pd.to_datetime(df['Datetime'])

mask_long_gap = pd.DataFrame(
    False, index=df.index, columns=pollutants
)

for _, g in df.groupby('StationId', sort=False):
    g = g.sort_values('Datetime')

    for col in pollutants:
        missing = g[col].isna()

        # Phân tách chuỗi khi đổi trạng thái hoặc thời gian bị gián đoạn
        breaks = (
            missing.ne(missing.shift(fill_value=False))
            | g['Datetime'].diff().ne(pd.Timedelta('1h'))
        )
        run_id = breaks.cumsum()

        for _, run in g.loc[missing].groupby(run_id[missing]):
            if len(run) / len(g) > 0.30:
                mask_long_gap.loc[run.index, col] = True

# Bảng thống kê
summary_gap = pd.DataFrame({
    'Số lượng đánh dấu': mask_long_gap.sum(),
    'Tỷ lệ (%)': (mask_long_gap.sum() / len(df) * 100).round(3)
})

display(summary_gap)


,Số lượng đánh dấu,Tỷ lệ (%)
PM2.5,227636,8.792
PM10,846015,32.676
NO,100765,3.892
NO2,83683,3.232
NOx,82938,3.203
NH3,989759,38.228
CO,83222,3.214
SO2,324796,12.545
O3,357797,13.819
Benzene,535766,20.693


In [31]:

# Lưu dữ liệu gốc trước khi impute
source = df[pollutants].copy()

month = df['Datetime'].dt.month
day = df['Datetime'].dt.day
hour = df['Datetime'].dt.hour
station = df['StationId']

group_levels = [
    [station, month, day, hour],
    [station, day, hour],
    [station, hour],
]

# Impute theo thứ tự ưu tiên
for col in pollutants:
    original_values = source[col]

    for keys in group_levels:
        missing = df[col].isna() & ~mask_long_gap[col]

        if not missing.any():
            break

        group_mean = original_values.groupby(keys).transform('mean')
        fill_mask = missing & group_mean.notna()

        df.loc[fill_mask, col] = group_mean.loc[fill_mask]

# Thống kê kết quả
imputed_mask = (
    source.isna()
    & df[pollutants].notna()
    & ~mask_long_gap
)

remaining_mask = df[pollutants].isna()

summary_impute = pd.DataFrame({
    'Số lượng đã impute': imputed_mask.sum(),
    'Đã impute (%)': (imputed_mask.sum() / len(df) * 100).round(3),
    'Số lượng còn trống': remaining_mask.sum(),
    'Còn trống (%)': (remaining_mask.sum() / len(df) * 100).round(3),
})

display(summary_impute)


,Số lượng đã impute,Đã impute (%),Số lượng còn trống,Còn trống (%)
PM2.5,431713,16.674,227636,8.792
PM10,280408,10.830,846015,32.676
NO,512070,19.778,100765,3.892
NO2,461017,17.806,83683,3.232
NOx,505896,19.540,82938,3.203
NH3,256960,9.925,989759,38.228
CO,534032,20.626,83222,3.214
SO2,439400,16.971,324796,12.545
O3,383842,14.825,357797,13.819
Benzene,542807,20.965,536522,20.722


### 3.2.5. Tính AQI
- Tính AQI bằng công thức [theo như link](https://www.kaggle.com/code/rohanrao/calculating-aqi-air-quality-index-tutorial)
- AQI tính ra sẽ được lưu vào cột `AQI_calculated`, `AQI_bucket_calculated` 

In [ ]:
import numpy as np
import pandas as pd

# 1. Sắp xếp theo thời gian trong từng StationId
_original_index = df.index
df = df.reset_index(drop=True).copy()
df["_row_order"] = np.arange(len(df))
df["_sort_time"] = pd.to_datetime(df["Datetime"], errors="coerce")
df = df.sort_values(
    ["StationId", "_sort_time", "_row_order"], kind="stable"
)

# 2. Rolling average / maximum theo từng trạm
rolling_specs = [
    ("PM10_24hr_avg",  "PM10",   24, 16, "mean"),
    ("PM2.5_24hr_avg", "PM2.5",  24, 16, "mean"),
    ("SO2_24hr_avg",   "SO2",    24, 16, "mean"),
    ("NOx_24hr_avg",   "NOx",    24, 16, "mean"),
    ("NH3_24hr_avg",   "NH3",    24, 16, "mean"),
    ("CO_8hr_max",     "CO",      8,  1, "max"),
    ("O3_8hr_max",     "O3",      8,  1, "max"),
]

for output, source, window, min_periods, operation in rolling_specs:
    rolling = df.groupby("StationId", sort=False)[source].rolling(
        window=window, min_periods=min_periods
    )
    result = getattr(rolling, operation)().droplevel(0)
    df[output] = result.reindex(df.index).to_numpy()


# 3. Hàm tính Sub-Index vector hóa bằng NumPy
def calc_subindex(values, concentrations, aqi_levels):
    x = pd.to_numeric(values, errors="coerce").to_numpy(
        dtype=float, na_value=np.nan
    )
    concentrations = np.asarray(concentrations, dtype=float)
    aqi_levels = np.asarray(aqi_levels, dtype=float)

    # Nội suy tuyến tính giữa các breakpoint
    result = np.interp(x, concentrations, aqi_levels)

    # Ngoại suy ngoài khoảng breakpoint
    low = x < concentrations[0]
    result[low] = aqi_levels[0] + (
        (x[low] - concentrations[0])
        * (aqi_levels[1] - aqi_levels[0])
        / (concentrations[1] - concentrations[0])
    )

    high = x > concentrations[-1]
    result[high] = aqi_levels[-1] + (
        (x[high] - concentrations[-1])
        * (aqi_levels[-1] - aqi_levels[-2])
        / (concentrations[-1] - concentrations[-2])
    )

    result[~np.isfinite(x)] = np.nan
    return pd.Series(result, index=values.index)


# Breakpoints nồng độ và AQI
breakpoints = {
    "PM2.5_SubIndex": (
        "PM2.5_24hr_avg",
        [0, 30, 60, 90, 120, 250],
        [0, 50, 100, 200, 300, 400],
    ),
    "PM10_SubIndex": (
        "PM10_24hr_avg",
        [0, 50, 100, 250, 350, 430],
        [0, 50, 100, 200, 300, 400],
    ),
    "SO2_SubIndex": (
        "SO2_24hr_avg",
        [0, 40, 80, 380, 800, 1600],
        [0, 50, 100, 200, 300, 400],
    ),
    "NOx_SubIndex": (
        "NOx_24hr_avg",
        [0, 40, 80, 180, 280, 400],
        [0, 50, 100, 200, 300, 400],
    ),
    "NH3_SubIndex": (
        "NH3_24hr_avg",
        [0, 200, 400, 800, 1200, 1800],
        [0, 50, 100, 200, 300, 400],
    ),
    "CO_SubIndex": (
        "CO_8hr_max",
        [0, 1, 2, 10, 17, 34],
        [0, 50, 100, 200, 300, 400],
    ),
    "O3_SubIndex": (
        "O3_8hr_max",
        [0, 50, 100, 168, 208, 748],
        [0, 50, 100, 200, 300, 400],
    ),
}

for output, (source, concentrations, levels) in breakpoints.items():
    df[output] = calc_subindex(df[source], concentrations, levels)


# 4. Tính AQI từ các Sub-Index
subindex_cols = list(breakpoints)

# Đếm số chất có dữ liệu hợp lệ, kể cả Sub-Index = 0
df["Checks"] = df[subindex_cols].notna().sum(axis=1)

df["AQI_calculated"] = df[subindex_cols].max(axis=1).round()

# Cần ít nhất 3 chất và có ít nhất một trong hai chỉ số bụi
pm_cols = ["PM2.5_SubIndex", "PM10_SubIndex"]
valid_pm = df[pm_cols].notna().any(axis=1)

df.loc[(df["Checks"] < 3) | ~valid_pm, "AQI_calculated"] = np.nan


# 5. Phân loại AQI không cần apply()
df["AQI_bucket_calculated"] = pd.cut(
    df["AQI_calculated"],
    bins=[-np.inf, 50, 100, 200, 300, 400, np.inf],
    labels=["Good", "Satisfactory", "Moderate", "Poor", "Very Poor", "Severe"],
).astype("string")


# 6. Khôi phục thứ tự dòng và index ban đầu
df = df.sort_values("_row_order", kind="stable")
df = df.drop(columns=["_row_order", "_sort_time"])
df.index = _original_index

df.loc[df["AQI_calculated"].notna()].head(5)

,StationId,Datetime,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,...,SO2_SubIndex,NOx_SubIndex,NH3_SubIndex,CO_SubIndex,O3_SubIndex,Checks,AQI_calculated,AQI_bucket_calculated,CO_8hr_avg,O3_8hr_avg
15,AP001,2017-11-25 08:00:00,38.478261,69.728261,6.094545,17.973913,14.686087,13.346522,0.565455,9.370455,...,13.885973,15.439538,2.775102,12.346591,86.551467,7,142.0,Moderate,0.246932,86.551467
16,AP001,2017-11-25 09:00:00,104.000000,148.500000,1.930000,23.000000,13.750000,9.800000,0.100000,15.300000,...,14.194151,15.542359,2.755978,7.909091,86.066467,7,148.0,Moderate,0.158182,86.066467
17,AP001,2017-11-25 10:00:00,94.500000,142.000000,1.330000,16.250000,9.750000,9.650000,0.100000,17.000000,...,14.586143,15.355978,2.736896,7.909091,92.620217,7,152.0,Moderate,0.158182,92.620217
18,AP001,2017-11-25 11:00:00,82.750000,126.500000,1.470000,14.830000,9.070000,9.700000,0.100000,15.400000,...,14.831609,15.144479,2.720481,7.909091,100.667599,7,153.0,Moderate,0.158182,100.453967
19,AP001,2017-11-25 12:00:00,79.000000,124.000000,1.875000,21.150000,15.530000,9.400000,0.100000,29.480000,...,15.932528,15.357880,2.701957,7.909091,115.437820,7,153.0,Moderate,0.158182,110.497717


### 3.3. Đánh giá
- So sánh AQI sau khi cleanning với AQI gốc

In [36]:
import pandas as pd
import plotly.express as px

# Chọn trạm
station_id = df['StationId'].dropna().unique()[0]  # đổi ID nếu cần

plot_df = df[df['StationId'] == station_id].copy()
plot_df['Datetime'] = pd.to_datetime(plot_df['Datetime'])
plot_df = plot_df.sort_values('Datetime')

long_df = plot_df.melt(
    id_vars='Datetime',
    value_vars=['AQI', 'AQI_calculated'],
    var_name='Loại AQI',
    value_name='AQI_value'
).dropna(subset=['AQI_value'])

long_df['Loại AQI'] = long_df['Loại AQI'].replace({
    'AQI': 'AQI gốc',
    'AQI_calculated': 'AQI tính lại'
})

fig = px.line(
    long_df,
    x='Datetime',
    y='AQI_value',
    color='Loại AQI',
    title=f'So sánh AQI gốc và AQI tính lại — Station {station_id}'
)

fig.update_layout(
    xaxis_title='Thời gian',
    yaxis_title='AQI',
    hovermode='x unified',
    xaxis=dict(rangeslider=dict(visible=True))
)

fig.show(config={
    'scrollZoom': True,
    'displaylogo': False
})

# 4.